# Train GPT-2 on StackOverflow (QueryGPT — Phase 1, JAX/Flax)

Train a **GPT-2–style causal LM** on preprocessed StackOverflow Q→A sequences using **JAX + Flax**.

**Inputs (from preprocessing):**
- `data/processed/tokenizer/tokenizer.json` — your BPE vocab (~16k)
- `data/processed/tokenized/train/chunk_*.parquet` — `input_ids` (length 1024)
- `data/processed/tokenized/val/chunk_*.parquet`
- `data/processed/tokenized/manifest.json`

**You implement:** model module, training loop, eval, logging, checkpointing.

```bash
pip install -r requirements-train-jax.txt
```

## 1. Configuration

In [ ]:
from pathlib import Path
import os
import tempfile
import yaml

# --- project root + params (must run before `import jax`) ---
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PARAMS_PATH = PROJECT_ROOT / "params.yml"
assert PARAMS_PATH.exists(), f"Missing config file: {PARAMS_PATH}"

with PARAMS_PATH.open() as f:
    cfg = yaml.safe_load(f)

runtime_cfg = cfg.get("runtime", {})

# HPC /tmp is often quota-limited; route all temp/cache writes to project disk.
_tmp_rel = runtime_cfg.get("tmp_dir", "artifacts/jax_tmp")
_xdg_rel = runtime_cfg.get("xdg_cache_dir", "artifacts/xdg_cache")
_cuda_rel = runtime_cfg.get("cuda_cache_dir", "artifacts/cuda_cache")
_jax_cache_rel = runtime_cfg.get("jax_compilation_cache_dir", "artifacts/jax_cache")

JAX_TMP_DIR = (PROJECT_ROOT / _tmp_rel).resolve()
XDG_CACHE_DIR = (PROJECT_ROOT / _xdg_rel).resolve()
CUDA_CACHE_DIR = (PROJECT_ROOT / _cuda_rel).resolve()
JAX_CACHE_DIR = (PROJECT_ROOT / _jax_cache_rel).resolve()

for _p in (JAX_TMP_DIR, XDG_CACHE_DIR, CUDA_CACHE_DIR, JAX_CACHE_DIR):
    _p.mkdir(parents=True, exist_ok=True)

for _var in ("TMPDIR", "TEMP", "TMP"):
    os.environ[_var] = str(JAX_TMP_DIR)
os.environ["XDG_CACHE_HOME"] = str(XDG_CACHE_DIR)
os.environ["CUDA_CACHE_PATH"] = str(CUDA_CACHE_DIR)
os.environ["JAX_COMPILATION_CACHE_DIR"] = str(JAX_CACHE_DIR)

# Python caches tempdir after first lookup; force it to follow TMPDIR.
tempfile.tempdir = str(JAX_TMP_DIR)

# IMPORTANT: set JAX runtime env BEFORE importing jax.
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("XLA_PYTHON_CLIENT_ALLOCATOR", "platform")

import jax

# --- data paths ---
PROCESSED_DIR = PROJECT_ROOT / cfg["paths"]["processed_dir"]
TOKENIZER_PATH = PROCESSED_DIR / cfg["paths"]["tokenizer_relpath"]
TRAIN_GLOB = str(PROCESSED_DIR / cfg["paths"]["train_glob"])
VAL_GLOB = str(PROCESSED_DIR / cfg["paths"]["val_glob"])
TEST_GLOB = str(PROCESSED_DIR / cfg["paths"].get("test_glob", "tokenized/test/chunk_*.parquet"))
MANIFEST_PATH = PROCESSED_DIR / cfg["paths"]["manifest_relpath"]
RUNS_ROOT = PROJECT_ROOT / cfg["paths"]["checkpoint_dir"]
METRICS_FILENAME = cfg["paths"].get("metrics_filename", "metrics.json")

# --- data ---
MAX_SEQ_LEN = int(cfg["data"]["max_seq_len"])
PAD_TOKEN = cfg["data"].get("pad_token", "<pad>")
ROW_BATCH_MULTIPLIER = int(cfg.get("dataloader", {}).get("row_batch_multiplier", 4))

# --- model ---
N_LAYER = int(cfg["model"]["n_layer"])
N_HEAD = int(cfg["model"]["n_head"])
N_EMBD = int(cfg["model"]["n_embd"])
DROPOUT = float(cfg["model"]["dropout"])

# --- training ---
GLOBAL_BATCH_SIZE = int(cfg["training"]["global_batch_size"])
GRAD_ACCUM_STEPS = int(cfg["training"].get("grad_accum_steps", 1))
LR = float(cfg["training"]["lr"])
WEIGHT_DECAY = float(cfg["training"]["weight_decay"])
MAX_STEPS = int(cfg["training"]["max_steps"])
WARMUP_STEPS = int(cfg["training"]["warmup_steps"])
LOG_EVERY = int(cfg["training"].get("log_every", 20))
EVAL_EVERY = int(cfg["training"]["eval_every"])
SAVE_EVERY = int(cfg["training"]["save_every"])
SEED = int(cfg["training"]["seed"])

# --- resume / run folder ---
RESUME_TRAINING = bool(cfg["resume"].get("enabled", False))
RESUME_CHECKPOINT = cfg["resume"].get("checkpoint", "latest")
RESUME_RUN_DIR = cfg["resume"].get("run_dir", "")

# --- generation ---
GEN_CONTEXT_WINDOW = int(cfg.get("generation", {}).get("context_window", 256))
GEN_MAX_NEW_TOKENS = int(cfg.get("generation", {}).get("max_new_tokens", 64))
GEN_TEMPERATURE = float(cfg.get("generation", {}).get("temperature", 0.8))
GEN_TOP_K = int(cfg.get("generation", {}).get("top_k", 40))
GEN_REPETITION_PENALTY = float(cfg.get("generation", {}).get("repetition_penalty", 1.15))
GEN_NO_REPEAT_NGRAM_SIZE = int(cfg.get("generation", {}).get("no_repeat_ngram_size", 3))
GEN_CHECKPOINT_PATH = cfg.get("generation", {}).get("checkpoint_path", "")

from datetime import datetime

if RESUME_TRAINING:
    if not RESUME_RUN_DIR:
        raise ValueError("resume.run_dir must be set in params.yml when resume.enabled=true")
    RUN_DIR = PROJECT_ROOT / RESUME_RUN_DIR
else:
    RUN_DIR = RUNS_ROOT / datetime.now().strftime("%Y%m%d_%H%M%S")

CHECKPOINT_DIR = RUN_DIR
METRICS_PATH = RUN_DIR / METRICS_FILENAME
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Persist a copy of config used for this run.
(RUN_DIR / "params_snapshot.yml").write_text(yaml.safe_dump(cfg, sort_keys=False))

# --- multi-device ---
USE_PMAP = bool(cfg.get("distributed", {}).get("use_pmap", True))
REQUIRE_NUM_DEVICES = int(cfg.get("distributed", {}).get("require_num_devices", 1))
NUM_DEVICES = jax.local_device_count()

if USE_PMAP:
    if NUM_DEVICES < REQUIRE_NUM_DEVICES:
        raise RuntimeError(
            f"USE_PMAP=True but only {NUM_DEVICES} device(s) visible. "
            f"Expected >= {REQUIRE_NUM_DEVICES}. Check CUDA_VISIBLE_DEVICES / JAX runtime."
        )
    assert GLOBAL_BATCH_SIZE % NUM_DEVICES == 0, (
        f"global_batch_size ({GLOBAL_BATCH_SIZE}) must be divisible by device_count ({NUM_DEVICES})"
    )

PER_DEVICE_BATCH_SIZE = GLOBAL_BATCH_SIZE // NUM_DEVICES if USE_PMAP else GLOBAL_BATCH_SIZE

print("Project:", PROJECT_ROOT)
print("Params:", PARAMS_PATH)
print("Runs root:", RUNS_ROOT)
print("Run dir:", RUN_DIR)
print("Metrics file:", METRICS_PATH)
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES", "<unset>"))
print("JAX devices:", jax.devices())
print(f"USE_PMAP={USE_PMAP}  NUM_DEVICES={NUM_DEVICES}  GLOBAL_BATCH_SIZE={GLOBAL_BATCH_SIZE}")
print("TMPDIR (JAX compile temp):", os.environ.get("TMPDIR"))
print("tempfile.gettempdir():", tempfile.gettempdir())
print("XDG_CACHE_HOME:", os.environ.get("XDG_CACHE_HOME"))
print("CUDA_CACHE_PATH:", os.environ.get("CUDA_CACHE_PATH"))
print("JAX_COMPILATION_CACHE_DIR:", os.environ.get("JAX_COMPILATION_CACHE_DIR"))
print("XLA_PYTHON_CLIENT_PREALLOCATE:", os.environ.get("XLA_PYTHON_CLIENT_PREALLOCATE"))
print("XLA_PYTHON_CLIENT_ALLOCATOR:", os.environ.get("XLA_PYTHON_CLIENT_ALLOCATOR"))
print("NOTE: restart kernel after changing runtime/tmp_dir or JAX env vars.")

## 2. Sanity check — verify preprocessed data

In [ ]:
import json
import glob

assert TOKENIZER_PATH.exists(), f"Missing {TOKENIZER_PATH}"
assert MANIFEST_PATH.exists(), f"Missing {MANIFEST_PATH}"

manifest = json.loads(MANIFEST_PATH.read_text())
train_chunks = sorted(glob.glob(TRAIN_GLOB))
val_chunks = sorted(glob.glob(VAL_GLOB))
test_chunks = sorted(glob.glob(TEST_GLOB))

print("manifest:", json.dumps(manifest, indent=2))
print(f"train chunks: {len(train_chunks)}  |  val chunks: {len(val_chunks)}  |  test chunks: {len(test_chunks)}")
print(f"expected train rows: {manifest['stats']['train']:,}")
print(f"expected val rows:   {manifest['stats']['val']:,}")
if 'test' in manifest.get('stats', {}):
    print(f"expected test rows:  {manifest['stats']['test']:,}")
assert manifest["max_seq_len"] == MAX_SEQ_LEN

## 3. Tokenizer

Load **your** BPE tokenizer from preprocessing. Resize GPT-2 embeddings to `vocab_size`.

In [ ]:
import re
from tokenizers import Tokenizer

hf_tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))
VOCAB_SIZE = hf_tokenizer.get_vocab_size()
PAD_ID = hf_tokenizer.token_to_id(PAD_TOKEN)
BOS_ID = hf_tokenizer.token_to_id("<bos>")
EOS_ID = hf_tokenizer.token_to_id("<eos>")
SEP_ID = hf_tokenizer.token_to_id("<sep>")

print("vocab_size:", VOCAB_SIZE)
print("pad_id:", PAD_ID, "bos_id:", BOS_ID, "eos_id:", EOS_ID, "sep_id:", SEP_ID)


def encode(text: str) -> list[int]:
    return hf_tokenizer.encode(text).ids


def _clean_bytelevel_artifacts(text: str) -> str:
    """Normalize byte-level BPE markers for human-readable output."""
    # Common byte-level markers from tokenizers BPE
    text = text.replace("Ġ", " ").replace("Ċ", "\n")
    # collapse repeated spaces and trim around punctuation
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\s+([,.;:!?])", r"\1", text)
    text = re.sub(r"\(\s+", "(", text)
    text = re.sub(r"\s+\)", ")", text)
    return text.strip()


def decode(ids: list[int], clean: bool = True) -> str:
    text = hf_tokenizer.decode(ids)
    return _clean_bytelevel_artifacts(text) if clean else text


def encode_batch(texts: list[str]) -> list[list[int]]:
    return [enc.ids for enc in hf_tokenizer.encode_batch(texts)]


def decode_batch(batch_ids: list[list[int]], clean: bool = True) -> list[str]:
    return [decode(ids, clean=clean) for ids in batch_ids]

## 4. Dataset & DataLoader

**Causal LM:** `labels = input_ids`, loss only on non-`pad` positions.

Implement / customize `SOChunkDataset` below (starter provided).

In [ ]:
import numpy as np
import pyarrow.parquet as pq

import jax
import jax.numpy as jnp


def _list_array_to_numpy(list_array, seq_len: int) -> np.ndarray:
    """Convert Arrow ListArray[int] -> writable np.ndarray[B, T]."""
    try:
        vals = np.asarray(list_array.values.to_numpy(), dtype=np.int32)
        # Arrow may return a read-only view; force a writable copy.
        return np.array(vals.reshape(len(list_array), seq_len), dtype=np.int32, copy=True)
    except Exception:
        # Fallback for environments where Arrow conversion differs
        return np.array(list_array.to_pylist(), dtype=np.int32, copy=True)


def iter_parquet_batches(
    chunk_paths: list[str],
    batch_size: int,
    *,
    repeat: bool = False,
    shuffle_files: bool = True,
    shuffle_rows: bool = True,
    drop_last: bool = True,
    seed: int = 42,
):
    """Stream tokenized parquet batches without loading all data into RAM."""
    rng = np.random.default_rng(seed)
    paths = list(chunk_paths)

    while True:
        if shuffle_files:
            rng.shuffle(paths)

        for path in paths:
            pf = pq.ParquetFile(path)
            row_batch_size = max(batch_size * ROW_BATCH_MULTIPLIER, 256)
            for rb in pf.iter_batches(batch_size=row_batch_size, columns=["input_ids"]):
                arr = _list_array_to_numpy(rb.column(0), MAX_SEQ_LEN)
                if shuffle_rows:
                    rng.shuffle(arr)
                for i in range(0, len(arr), batch_size):
                    b = arr[i : i + batch_size]
                    if len(b) < batch_size and drop_last:
                        continue
                    yield b

        if not repeat:
            break


def make_batch(np_ids: np.ndarray) -> dict[str, jnp.ndarray]:
    ids = jnp.asarray(np_ids, dtype=jnp.int32)
    return {"input_ids": ids, "labels": ids}


def shard_batch(batch: dict[str, jnp.ndarray]) -> dict[str, jnp.ndarray]:
    """Reshape (global_batch, ...) -> (num_devices, per_device_batch, ...)."""
    if not USE_PMAP:
        return batch

    def _reshape(x):
        x = np.asarray(x)
        new_shape = (NUM_DEVICES, PER_DEVICE_BATCH_SIZE) + x.shape[1:]
        return jnp.asarray(x.reshape(new_shape), dtype=jnp.int32)

    return {k: _reshape(v) for k, v in batch.items()}


# Smoke test
# batch_np = next(iter(iter_parquet_batches(train_chunks[:1], batch_size=GLOBAL_BATCH_SIZE, repeat=False)))
# batch = make_batch(batch_np)
# sharded = shard_batch(batch)
# {k: v.shape for k, v in sharded.items()}

### 4b. Streaming iterator (implemented)

This iterator is memory-safe for large corpora:
- shuffles chunk file order each epoch
- shuffles rows within each Arrow batch
- supports repeat mode for long training loops

## 5. Model — GPT-2 in Flax

Below is a complete GPT-2-style decoder implemented with `flax.linen`:
- learned token + positional embeddings
- pre-LN transformer blocks
- causal self-attention
- MLP with GELU
- tied LM head (token embedding matrix transpose)

In [ ]:
import flax.linen as nn
import jax
import jax.numpy as jnp


class CausalSelfAttention(nn.Module):
    n_embd: int
    n_head: int
    dropout: float

    @nn.compact
    def __call__(self, x, train: bool):
        B, T, C = x.shape
        head_dim = C // self.n_head
        qkv = nn.Dense(3 * C, use_bias=True, name="qkv_proj")(x)
        q, k, v = jnp.split(qkv, 3, axis=-1)

        def split_heads(t):
            return t.reshape(B, T, self.n_head, head_dim).transpose(0, 2, 1, 3)

        q = split_heads(q)
        k = split_heads(k)
        v = split_heads(v)

        att = jnp.einsum("bhqd,bhkd->bhqk", q, k) / jnp.sqrt(head_dim)

        causal = jnp.tril(jnp.ones((T, T), dtype=jnp.bool_))[None, None, :, :]
        att = jnp.where(causal, att, -1e10)
        att = nn.softmax(att, axis=-1)
        att = nn.Dropout(rate=self.dropout)(att, deterministic=not train)

        y = jnp.einsum("bhqk,bhkd->bhqd", att, v)
        y = y.transpose(0, 2, 1, 3).reshape(B, T, C)
        y = nn.Dense(C, use_bias=True, name="out_proj")(y)
        y = nn.Dropout(rate=self.dropout)(y, deterministic=not train)
        return y


class MLP(nn.Module):
    n_embd: int
    dropout: float

    @nn.compact
    def __call__(self, x, train: bool):
        x = nn.Dense(4 * self.n_embd, name="fc")(x)
        x = nn.gelu(x, approximate=False)
        x = nn.Dense(self.n_embd, name="proj")(x)
        x = nn.Dropout(rate=self.dropout)(x, deterministic=not train)
        return x


class Block(nn.Module):
    n_embd: int
    n_head: int
    dropout: float

    @nn.compact
    def __call__(self, x, train: bool):
        x = x + CausalSelfAttention(self.n_embd, self.n_head, self.dropout, name="attn")(
            nn.LayerNorm(name="ln_1")(x),
            train=train,
        )
        x = x + MLP(self.n_embd, self.dropout, name="mlp")(
            nn.LayerNorm(name="ln_2")(x),
            train=train,
        )
        return x


class GPT2LM(nn.Module):
    vocab_size: int
    n_layer: int
    n_head: int
    n_embd: int
    max_seq_len: int
    dropout: float = 0.1

    @nn.compact
    def __call__(self, input_ids: jnp.ndarray, train: bool):
        B, T = input_ids.shape
        tok_emb = nn.Embed(self.vocab_size, self.n_embd, name="wte")
        pos_emb = self.param(
            "wpe",
            nn.initializers.normal(stddev=0.02),
            (self.max_seq_len, self.n_embd),
        )

        x = tok_emb(input_ids) + pos_emb[:T][None, :, :]
        x = nn.Dropout(rate=self.dropout)(x, deterministic=not train)

        for i in range(self.n_layer):
            x = Block(self.n_embd, self.n_head, self.dropout, name=f"h_{i}")(x, train=train)

        x = nn.LayerNorm(name="ln_f")(x)

        # Weight tying: logits = hidden @ token_embedding^T
        logits = jnp.einsum("btc,vc->btv", x, tok_emb.embedding)
        return logits


rng = jax.random.PRNGKey(SEED)
rng, init_rng, drop_rng = jax.random.split(rng, 3)

model = GPT2LM(
    vocab_size=VOCAB_SIZE,
    n_layer=N_LAYER,
    n_head=N_HEAD,
    n_embd=N_EMBD,
    max_seq_len=MAX_SEQ_LEN,
    dropout=DROPOUT,
)

dummy_ids = jnp.ones((1, MAX_SEQ_LEN), dtype=jnp.int32)
variables = model.init({"params": init_rng, "dropout": drop_rng}, dummy_ids, train=True)
params = variables["params"]

n_params = sum(x.size for x in jax.tree_util.tree_leaves(params))
print(f"Model params: {n_params/1e6:.2f}M")

## 6. Loss, optimizer, scheduler

Standard causal LM: shift logits/labels by 1. Ignore `pad` in loss (`labels[labels == pad_id] = -100`).

In [ ]:
import optax
from flax.training import train_state
from flax import jax_utils


def causal_lm_loss(logits: jnp.ndarray, labels: jnp.ndarray, pad_id: int = PAD_ID) -> jnp.ndarray:
    """Cross-entropy next-token loss with pad masking."""
    shift_logits = logits[:, :-1, :]
    shift_labels = labels[:, 1:]
    mask = (shift_labels != pad_id).astype(jnp.float32)

    token_loss = optax.softmax_cross_entropy_with_integer_labels(shift_logits, shift_labels)
    token_loss = token_loss * mask
    return token_loss.sum() / jnp.maximum(mask.sum(), 1.0)


lr_schedule = optax.warmup_cosine_decay_schedule(
    init_value=0.0,
    peak_value=LR,
    warmup_steps=WARMUP_STEPS,
    decay_steps=MAX_STEPS,
    end_value=LR * 0.1,
)

tx = optax.chain(
    optax.clip_by_global_norm(1.0),
    optax.adamw(learning_rate=lr_schedule, weight_decay=WEIGHT_DECAY, b1=0.9, b2=0.95),
)

state = train_state.TrainState.create(
    apply_fn=model.apply,
    params=params,
    tx=tx,
)

if USE_PMAP:
    state = jax_utils.replicate(state)

print("Initial LR:", float(lr_schedule(0)))

## 7. Checkpointing (run before training)

Define checkpoint helpers **before** the training loop so cell-by-cell execution works without errors.

In [ ]:
import shutil
import orbax.checkpoint as ocp
from flax import jax_utils

checkpointer = ocp.PyTreeCheckpointer()


def _state_to_host(state, use_pmap: bool):
    return jax_utils.unreplicate(state) if use_pmap else state


def save_ckpt(path: Path, state, rng_key, step: int, use_pmap: bool = False):
    host_state = _state_to_host(state, use_pmap)
    payload = {
        "params": jax.device_get(host_state.params),
        "opt_state": jax.device_get(host_state.opt_state),
        "step": int(step),
        "rng": jax.device_get(rng_key),
        "model_config": {
            "vocab_size": VOCAB_SIZE,
            "n_layer": N_LAYER,
            "n_head": N_HEAD,
            "n_embd": N_EMBD,
            "max_seq_len": MAX_SEQ_LEN,
        },
    }
    if path.exists():
        shutil.rmtree(path)
    checkpointer.save(str(path), payload)
    print("saved", path)


def restore_ckpt(path: Path, state, use_pmap: bool = False):
    restored = checkpointer.restore(str(path))
    host_state = _state_to_host(state, use_pmap)
    host_state = host_state.replace(params=restored["params"], opt_state=restored["opt_state"])
    state = jax_utils.replicate(host_state) if use_pmap else host_state
    step = int(restored["step"])
    rng_key = restored["rng"]
    print(f"restored {path} @ step {step}")
    return state, rng_key, step

## 7. Training loop (JAX)

The loop below runs fully in JAX with `@jax.jit` train/eval steps:
- fetch numpy batch from parquet iterator
- convert to JAX arrays
- run `train_step(state, batch, dropout_rng)`
- evaluate perplexity every `EVAL_EVERY`
- checkpoint via Orbax every `SAVE_EVERY`

In [ ]:
from functools import partial

@jax.jit
def train_step_single(state, batch, dropout_rng):
    def loss_fn(params):
        logits = state.apply_fn(
            {"params": params},
            batch["input_ids"],
            train=True,
            rngs={"dropout": dropout_rng},
        )
        return causal_lm_loss(logits, batch["labels"], PAD_ID)

    loss, grads = jax.value_and_grad(loss_fn)(state.params)
    state = state.apply_gradients(grads=grads)
    return state, loss


@partial(jax.pmap, axis_name="devices")
def train_step_pmap(state, batch, dropout_rng):
    def loss_fn(params):
        logits = state.apply_fn(
            {"params": params},
            batch["input_ids"],
            train=True,
            rngs={"dropout": dropout_rng},
        )
        return causal_lm_loss(logits, batch["labels"], PAD_ID)

    loss, grads = jax.value_and_grad(loss_fn)(state.params)
    loss = jax.lax.pmean(loss, axis_name="devices")
    grads = jax.lax.pmean(grads, axis_name="devices")
    state = state.apply_gradients(grads=grads)
    return state, loss


@jax.jit
def eval_step_single(params, batch):
    logits = model.apply({"params": params}, batch["input_ids"], train=False)
    return causal_lm_loss(logits, batch["labels"], PAD_ID)


@partial(jax.pmap, axis_name="devices")
def eval_step_pmap(params, batch):
    logits = model.apply({"params": params}, batch["input_ids"], train=False)
    loss = causal_lm_loss(logits, batch["labels"], PAD_ID)
    return jax.lax.pmean(loss, axis_name="devices")


def _append_metric(record: dict):
    METRICS_PATH.parent.mkdir(parents=True, exist_ok=True)
    if METRICS_PATH.exists():
        data = json.loads(METRICS_PATH.read_text())
    else:
        data = []
    data.append(record)
    METRICS_PATH.write_text(json.dumps(data, indent=2))


def _evaluate_perplexity_local(state, val_iter, max_batches: int = 50):
    """Local eval helper so run_training works even if later cells weren't run yet."""
    losses = []
    for i, batch_np in enumerate(val_iter):
        if i >= max_batches:
            break
        batch = make_batch(batch_np)

        if USE_PMAP:
            batch = shard_batch(batch)
            loss = eval_step_pmap(state.params, batch)
            losses.append(float(jax.device_get(loss)[0]))
        else:
            loss = eval_step_single(state.params, batch)
            losses.append(float(loss))

    if not losses:
        return 0.0, 1.0

    mean_loss = float(np.mean(losses))
    return mean_loss, float(np.exp(mean_loss))


from tqdm.auto import tqdm


def run_training(num_steps: int = MAX_STEPS, eval_batches: int = 64):
    global state, rng

    if "save_ckpt" not in globals():
        raise RuntimeError(
            "save_ckpt is not defined. Run the checkpointing cell before training so checkpoints are written."
        )

    start_step = 0
    if RESUME_TRAINING and "restore_ckpt" in globals():
        ckpt_path = CHECKPOINT_DIR / RESUME_CHECKPOINT
        if ckpt_path.exists():
            state, rng, start_step = restore_ckpt(ckpt_path, state, use_pmap=USE_PMAP)

    train_iter = iter_parquet_batches(
        train_chunks,
        batch_size=GLOBAL_BATCH_SIZE,
        repeat=True,
        shuffle_files=True,
        shuffle_rows=True,
        drop_last=True,
        seed=SEED,
    )

    running = []
    pbar = tqdm(range(start_step + 1, num_steps + 1), desc="Training", unit="step", dynamic_ncols=True)

    for step in pbar:
        batch_np = next(train_iter)
        batch = make_batch(batch_np)

        if USE_PMAP:
            batch = shard_batch(batch)
            if step == start_step + 1:
                pbar.write(f"pmap batch shape: {batch['input_ids'].shape} (num_devices, per_device_batch, seq)")
            rng, base = jax.random.split(rng)
            step_rng = jax.random.split(base, NUM_DEVICES)
            state, loss = train_step_pmap(state, batch, step_rng)
            loss_scalar = float(jax.device_get(loss)[0])
            state_step = int(jax.device_get(state.step)[0])
        else:
            rng, step_rng = jax.random.split(rng)
            state, loss = train_step_single(state, batch, step_rng)
            loss_scalar = float(loss)
            state_step = int(state.step)

        running.append(loss_scalar)

        if step % LOG_EVERY == 0:
            mean_loss = float(np.mean(running[-LOG_EVERY:]))
            lr_now = float(lr_schedule(state_step))
            train_ppl = float(np.exp(mean_loss))
            pbar.set_postfix(loss=f"{mean_loss:.4f}", ppl=f"{train_ppl:.2f}", lr=f"{lr_now:.2e}")
            _append_metric({"step": step, "train_loss": mean_loss, "train_ppl": train_ppl, "lr": lr_now})

        if step % EVAL_EVERY == 0:
            val_iter = iter_parquet_batches(
                val_chunks,
                batch_size=GLOBAL_BATCH_SIZE,
                repeat=False,
                shuffle_files=False,
                shuffle_rows=False,
                drop_last=False,
                seed=SEED,
            )
            val_loss, val_ppl = _evaluate_perplexity_local(state, val_iter, max_batches=eval_batches)
            pbar.write(f"[eval] step={step:6d} | val_loss={val_loss:.4f} | val_ppl={val_ppl:.2f}")
            _append_metric({"step": step, "val_loss": val_loss, "val_ppl": val_ppl})

        if step % SAVE_EVERY == 0 and "save_ckpt" in globals():
            save_ckpt(CHECKPOINT_DIR / f"step_{step}", state, rng, step, use_pmap=USE_PMAP)
            save_ckpt(CHECKPOINT_DIR / "latest", state, rng, step, use_pmap=USE_PMAP)


# First run small to verify:
# run_training(num_steps=200, eval_batches=16)
# Full run:
# run_training(num_steps=MAX_STEPS, eval_batches=64)

## 8. Validation — perplexity

In [ ]:
def evaluate_perplexity(state, val_iter, max_batches: int = 50):
    """Compute mean loss and perplexity over validation batches."""
    losses = []
    for i, batch_np in enumerate(val_iter):
        if i >= max_batches:
            break
        batch = make_batch(batch_np)

        if USE_PMAP:
            batch = shard_batch(batch)
            loss = eval_step_pmap(state.params, batch)
            losses.append(jax.device_get(loss)[0])
        else:
            loss = eval_step_single(state.params, batch)
            losses.append(float(loss))

    if not losses:
        return 0.0, 1.0

    mean_loss = float(np.mean(losses))
    ppl = float(np.exp(mean_loss))
    return mean_loss, ppl


# Quick check:
# val_iter = iter_parquet_batches(val_chunks, batch_size=GLOBAL_BATCH_SIZE, repeat=False, shuffle_files=False, shuffle_rows=False, drop_last=False)
# val_loss, val_ppl = evaluate_perplexity(state, val_iter, max_batches=10)
# print(f"val_loss={val_loss:.4f} val_ppl={val_ppl:.2f}")

## 9. Checkpointing (already defined above)

Checkpoint helpers are defined before training in Section 7 so running cells top-to-bottom works cleanly.

In [ ]:
# Checkpoint helpers moved to Section 7 (before training).
# Keep this cell as a no-op for backward compatibility with older runs.
print("Checkpoint functions are already defined above.")

## 10. Generation smoke test

After training, decode a few tokens from a prompt to sanity-check the model.

In [ ]:
import time
from pathlib import Path


def _apply_repetition_penalty(logits: jnp.ndarray, generated_ids: list[int], penalty: float) -> jnp.ndarray:
    if penalty <= 1.0 or not generated_ids:
        return logits
    # Penalize already-generated tokens.
    for tok in set(generated_ids):
        val = logits[tok]
        logits = logits.at[tok].set(jnp.where(val > 0, val / penalty, val * penalty))
    return logits


def _ban_no_repeat_ngram_logits(logits: jnp.ndarray, generated_ids: list[int], n: int) -> jnp.ndarray:
    if n <= 1 or len(generated_ids) < n - 1:
        return logits
    prefix = tuple(generated_ids[-(n - 1):])
    banned = set()
    for i in range(len(generated_ids) - n + 1):
        gram = tuple(generated_ids[i : i + n])
        if gram[:-1] == prefix:
            banned.add(gram[-1])
    if banned:
        idx = jnp.asarray(list(banned), dtype=jnp.int32)
        logits = logits.at[idx].set(-1e10)
    return logits


def sample_next_token(
    logits_last: jnp.ndarray,
    rng_key,
    *,
    generated_ids: list[int],
    temperature: float = 1.0,
    top_k: int = 50,
    repetition_penalty: float = 1.0,
    no_repeat_ngram_size: int = 0,
):
    """Sample one token from logits vector (V,) with anti-repetition controls."""
    logits = logits_last
    logits = _apply_repetition_penalty(logits, generated_ids, repetition_penalty)
    logits = _ban_no_repeat_ngram_logits(logits, generated_ids, no_repeat_ngram_size)

    if temperature <= 0:
        return int(jnp.argmax(logits)), rng_key

    logits = logits / temperature
    if top_k is not None and top_k > 0:
        top_vals, top_idx = jax.lax.top_k(logits, k=min(top_k, logits.shape[-1]))
        probs = jax.nn.softmax(top_vals)
        rng_key, sub = jax.random.split(rng_key)
        choice = jax.random.categorical(sub, jnp.log(probs))
        token_id = int(top_idx[choice])
        return token_id, rng_key

    probs = jax.nn.softmax(logits)
    rng_key, sub = jax.random.split(rng_key)
    token_id = int(jax.random.categorical(sub, jnp.log(probs)))
    return token_id, rng_key


@jax.jit
def _next_logits(params, input_ids):
    """Fast next-token logits with fixed input shape [1, context_window]."""
    logits = model.apply({"params": params}, input_ids, train=False)
    return logits[:, -1, :]


def _build_fixed_context(ids: list[int], context_window: int) -> jnp.ndarray:
    ctx = ids[-context_window:]
    if len(ctx) < context_window:
        ctx = [PAD_ID] * (context_window - len(ctx)) + ctx
    return jnp.asarray([ctx], dtype=jnp.int32)


def _resolve_generation_checkpoint_path(path_like: str | None):
    if path_like is None or path_like == "":
        if GEN_CHECKPOINT_PATH:
            path_like = GEN_CHECKPOINT_PATH
        else:
            return None

    p = Path(path_like)
    if not p.is_absolute():
        p = PROJECT_ROOT / p
    return p


def load_state_for_generation(checkpoint_path: str | None = None):
    """Load checkpoint into current state skeleton for generation."""
    ckpt_path = _resolve_generation_checkpoint_path(checkpoint_path)
    if ckpt_path is None:
        return state
    if not ckpt_path.exists():
        raise FileNotFoundError(f"Checkpoint path not found: {ckpt_path}")

    loaded_state, _, loaded_step = restore_ckpt(ckpt_path, state, use_pmap=USE_PMAP)
    print(f"Loaded generation checkpoint from: {ckpt_path} (step={loaded_step})")
    return loaded_state


def generate_text(
    state,
    prompt: str,
    max_new_tokens: int | None = None,
    temperature: float | None = None,
    top_k: int | None = None,
    repetition_penalty: float | None = None,
    no_repeat_ngram_size: int | None = None,
    context_window: int | None = None,
    seed: int = 0,
    show_timing: bool = True,
    checkpoint_path: str | None = None,
):
    """Fast generation path (JIT + fixed context window)."""
    gen_state = load_state_for_generation(checkpoint_path)
    host_state = _state_to_host(gen_state, USE_PMAP)
    ids = encode(prompt)

    if max_new_tokens is None:
        max_new_tokens = GEN_MAX_NEW_TOKENS
    if temperature is None:
        temperature = GEN_TEMPERATURE
    if top_k is None:
        top_k = GEN_TOP_K
    if repetition_penalty is None:
        repetition_penalty = GEN_REPETITION_PENALTY
    if no_repeat_ngram_size is None:
        no_repeat_ngram_size = GEN_NO_REPEAT_NGRAM_SIZE
    if context_window is None:
        context_window = min(GEN_CONTEXT_WINDOW, MAX_SEQ_LEN)

    rng_key = jax.random.PRNGKey(seed)

    # Warmup compile once for this context shape.
    x0 = _build_fixed_context(ids, context_window)
    _ = _next_logits(host_state.params, x0)

    prompt_len = len(ids)
    t0 = time.time()
    for _step in range(max_new_tokens):
        x = _build_fixed_context(ids, context_window)
        logits_last = _next_logits(host_state.params, x)[0]
        next_id, rng_key = sample_next_token(
            logits_last,
            rng_key,
            generated_ids=ids,
            temperature=temperature,
            top_k=top_k,
            repetition_penalty=repetition_penalty,
            no_repeat_ngram_size=no_repeat_ngram_size,
        )
        ids.append(next_id)
        if EOS_ID is not None and next_id == EOS_ID:
            break

    if show_timing:
        dt = time.time() - t0
        gen_toks = max(1, len(ids) - prompt_len)
        print(f"generated {gen_toks} tokens in {dt:.2f}s ({gen_toks/dt:.2f} tok/s), context_window={context_window}")

    return decode(ids)


# Fast smoke test after training:
# prompt = "<bos> Question:\nHow do I fix a MemoryError in Python?\n<sep> Answer:\n"
# print(generate_text(
#     state,
#     prompt,
#     checkpoint_path="checkpoints/gpt2-so-jax/20260526_220000/latest",
#     temperature=0.3,
#     top_k=20,
#     repetition_penalty=1.2,
#     no_repeat_ngram_size=3,
# ))

## 11. Roadmap (after base LM works)

| Phase | Task |
|-------|------|
| **Done** | Preprocess SO → `tokenized/*/chunk_*.parquet` |
| **Now** | Train GPT-2 (this JAX notebook) |
| **Next** | Wikipedia + FAISS RAG index |
| **Next** | Compare generation **with vs without** retrieval |
| **Next** | Streamlit chatbot demo |

---

### Suggested order for iteration

1. Run end-to-end for 200–500 steps (`MAX_STEPS`) to sanity-check loss
2. Add richer validation (more batches, accuracy-style probes)
3. Add multi-device training (`pmap` or `pjit`) once single-device is stable
4. Add W&B logging and better checkpoint retention policy